In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from xgboost import XGBClassifier

import joblib

In [2]:
df = pd.read_csv("Disease_and_Symptoms_Preprocessed.csv")

In [3]:
df.head()

,diseases,anxiety and nervousness,depression,shortness of breath,depressive or psychotic symptoms,sharp chest pain,dizziness,insomnia,abnormal involuntary movements,chest tightness,...,stuttering or stammering,problems with orgasm,nose deformity,lump over jaw,sore in nose,hip weakness,back swelling,ankle stiffness or tightness,ankle weakness,neck weakness
0,panic disorder,1,0,1,1,0,0,0,0,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,panic disorder,0,0,1,1,0,1,1,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,panic disorder,1,1,1,1,0,1,1,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,panic disorder,1,0,0,1,0,1,1,1,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,panic disorder,1,1,0,0,0,0,1,1,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
df.shape

(50238, 378)

In [5]:
df.isnull().sum().sum()

np.int64(268)

In [6]:
missing = df.isnull().sum()
missing = missing[missing > 0]
print(missing)

ear pain                        1
jaw swelling                    1
mouth dryness                   1
neck swelling                   1
knee pain                       1
                               ..
hip weakness                    1
back swelling                   1
ankle stiffness or tightness    1
ankle weakness                  1
neck weakness                   1
Length: 268, dtype: int64


In [7]:
df = df.fillna(0)

In [8]:
print(df.isnull().sum().sum())

0


In [9]:
X = df.drop("diseases", axis=1)
y = df["diseases"]

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

Features Shape: (50238, 377)
Target Shape: (50238,)


In [10]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("Total Diseases:", len(label_encoder.classes_))
print("First 10 Encoded Labels:", y_encoded[:10])

Total Diseases: 227
First 10 Encoded Labels: [159 159 159 159 159 159 159 159 159 159]


In [11]:
disease_counts = df["diseases"].value_counts()

print(disease_counts)

diseases
vulvodynia                        1218
complex regional pain syndrome    1217
spondylosis                       1216
hypoglycemia                      1215
infectious gastroenteritis        1212
                                  ... 
pulmonic valve disease               1
diabetes                             1
open wound of the chest              1
huntington disease                   1
insulin overdose                     1
Name: count, Length: 227, dtype: int64


In [12]:
print(disease_counts[disease_counts < 2])

diseases
turner syndrome            1
dengue fever               1
hashimoto thyroiditis      1
cryptococcosis             1
pulmonic valve disease     1
diabetes                   1
open wound of the chest    1
huntington disease         1
insulin overdose           1
Name: count, dtype: int64


In [13]:
valid_diseases = disease_counts[disease_counts >= 2].index

df = df[df["diseases"].isin(valid_diseases)]

print(df.shape)

(50229, 378)


In [14]:
X = df.drop("diseases", axis=1)
y = df["diseases"]

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

Features Shape: (50229, 377)
Target Shape: (50229,)


In [15]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("Total Diseases:", len(label_encoder.classes_))

Total Diseases: 218


In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print("Training Shape :", X_train.shape)
print("Testing Shape  :", X_test.shape)

Training Shape : (40183, 377)
Testing Shape  : (10046, 377)


In [26]:
model = XGBClassifier(
    objective="multi:softprob",
    num_class=len(label_encoder.classes_),
    n_estimators=20,
    max_depth=4,
    learning_rate=0.3,
    tree_method="hist",
    random_state=42,
    eval_metric="mlogloss",
    n_jobs=2
)

In [27]:
model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.3, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=20, n_jobs=2, num_class=218, ...)

In [23]:
import xgboost
print(xgboost.__version__)

3.3.0


In [24]:
import multiprocessing

print("CPU Cores:", multiprocessing.cpu_count())

CPU Cores: 2


In [25]:
print(X_train.shape)
print(y_train.shape)
print(len(label_encoder.classes_))

(40183, 377)
(40183,)
218


In [28]:
y_pred = model.predict(X_test)

In [29]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 0.9034441568783595


In [30]:
from sklearn.metrics import precision_score, recall_score, f1_score

precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
recall = recall_score(y_test, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

Accuracy : 0.9034441568783595
Precision: 0.9016312292789469
Recall   : 0.9034441568783595
F1 Score : 0.9011397646434735


In [32]:
from sklearn.metrics import classification_report
import numpy as np

labels = np.unique(np.concatenate([y_test, y_pred]))

print(classification_report(
    y_test,
    y_pred,
    labels=labels,
    target_names=label_encoder.inverse_transform(labels),
    zero_division=0
))

                                                          precision    recall  f1-score   support

                                        abdominal hernia       0.98      0.98      0.98        53
                                     abscess of the lung       0.00      0.00      0.00         1
                                               achalasia       0.50      0.60      0.55         5
                                       actinic keratosis       0.92      0.93      0.92       162
                                      acute bronchospasm       0.85      0.78      0.81       162
                                          acute glaucoma       0.63      0.63      0.63        30
                                      acute otitis media       0.93      0.93      0.93       107
                                      acute pancreatitis       0.97      0.98      0.97       242
                                         acute sinusitis       0.91      0.90      0.91       166
                   

In [33]:
from sklearn.metrics import classification_report
import numpy as np

labels = np.unique(np.concatenate([y_test, y_pred]))

print(classification_report(
    y_test,
    y_pred,
    labels=labels,
    target_names=label_encoder.inverse_transform(labels),
    zero_division=0
))

                                                          precision    recall  f1-score   support

                                        abdominal hernia       0.98      0.98      0.98        53
                                     abscess of the lung       0.00      0.00      0.00         1
                                               achalasia       0.50      0.60      0.55         5
                                       actinic keratosis       0.92      0.93      0.92       162
                                      acute bronchospasm       0.85      0.78      0.81       162
                                          acute glaucoma       0.63      0.63      0.63        30
                                      acute otitis media       0.93      0.93      0.93       107
                                      acute pancreatitis       0.97      0.98      0.97       242
                                         acute sinusitis       0.91      0.90      0.91       166
                   

In [34]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix Shape:", cm.shape)

Confusion Matrix Shape: (212, 212)


In [35]:
print(cm)

[[ 52   0   0 ...   0   0   0]
 [  0   0   0 ...   0   0   0]
 [  0   0   3 ...   0   0   0]
 ...
 [  0   0   0 ...   2   0   0]
 [  0   0   0 ...   0   0   0]
 [  0   0   0 ...   0   0 238]]


In [36]:
import pandas as pd

importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

importance.head(20)

,Feature,Importance
261,eyelid swelling,0.080678
41,white discharge from eye,0.072667
326,leg weakness,0.066000
235,pulling at ears,0.057354
248,"muscle cramps, contractures, or spasms",0.057053
156,itchy eyelid,0.051215
105,symptoms of infants,0.049000
315,antisocial behavior,0.048575
135,difficulty eating,0.047233
187,swollen abdomen,0.047115


In [37]:
import joblib

joblib.dump(model, "xgboost_model.pkl")

print("Model Saved Successfully!")

Model Saved Successfully!


In [38]:
joblib.dump(label_encoder, "label_encoder.pkl")

print("Label Encoder Saved Successfully!")

Label Encoder Saved Successfully!


In [39]:
sample = X.iloc[[0]]

prediction = model.predict(sample)

predicted_disease = label_encoder.inverse_transform(prediction)

print("Predicted Disease:", predicted_disease[0])

Predicted Disease: panic disorder


In [40]:
actual = y.iloc[0]

print("Actual Disease   :", actual)
print("Predicted Disease:", predicted_disease[0])

Actual Disease   : panic disorder
Predicted Disease: panic disorder
